In [1]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
TICKERS = ["RYAAY", "LHA.DE", "AF.PA", "IAG.L"]


In [7]:
def _get_row(df, candidates):
    """Find the first matching row label from a list of possible names"""
    
    for name in candidates:
        if name in df.index:
            return df.loc[name]
    return None
 

In [10]:
def fetch_company_data(ticker):
    """Pull balance sheet + income statement for one ticker and return"""
    t = yf.Ticker(ticker)
    bs = t.balance_sheet
    inc = t.financials
 
    if bs.empty or inc.empty:
        raise ValueError(f"No data returned for {ticker} (check ticker or network access)")
 
    # most recent year = first column
    latest_bs = bs.iloc[:, 0]
    latest_inc = inc.iloc[:, 0]
 
    data = {
        "ticker": ticker,
        "receivables": _get_row(bs, ["Accounts Receivable", "Receivables", "Net Receivables"]),
        "payables": _get_row(bs, ["Accounts Payable", "Payables", "Payables And Accrued Expenses"]),
        "inventory": _get_row(bs, ["Inventory", "Inventories"]),
        "current_assets": _get_row(bs, ["Current Assets", "Total Current Assets"]),
        "current_liabilities": _get_row(bs, ["Current Liabilities", "Total Current Liabilities"]),
        "cash": _get_row(bs, ["Cash And Cash Equivalents", "Cash Cash Equivalents And Short Term Investments"]),
        "revenue": _get_row(inc, ["Total Revenue", "Revenue"]),
        "cogs": _get_row(inc, ["Cost Of Revenue", "Reconciled Cost Of Revenue", "Cost Of Goods Sold"]),
    }
    return data
 
 
def calculate_ratios(data, days=365):
    """Compute DSO, DPO, DIO, CCC, current/quick/cash ratio from raw line items.
    Uses the most recent fiscal year (single-period average, not avg of two years,
    to keep it simple for a first-pass model)."""
    ticker = data["ticker"]
 
    def col0(x):
        # each item is a pandas Series across years - take latest (index 0)
        return x.iloc[0] if x is not None and len(x) > 0 else None
 
    receivables = col0(data["receivables"])
    payables = col0(data["payables"])
    inventory = col0(data["inventory"])
    current_assets = col0(data["current_assets"])
    current_liabilities = col0(data["current_liabilities"])
    cash = col0(data["cash"])
    revenue = col0(data["revenue"])
    cogs = col0(data["cogs"])
 
    results = {"ticker": ticker}
 
    results["DSO"] = round(receivables / revenue * days, 1) if receivables and revenue else None
    results["DPO"] = round(payables / cogs * days, 1) if payables and cogs else None
    results["DIO"] = round(inventory / cogs * days, 1) if inventory and cogs else None
 
    if results["DSO"] is not None and results["DIO"] is not None and results["DPO"] is not None:
        results["CCC"] = round(results["DSO"] + results["DIO"] - results["DPO"], 1)
    else:
        results["CCC"] = None
 
    results["Current Ratio"] = round(current_assets / current_liabilities, 2) if current_assets and current_liabilities else None
    results["Quick Ratio"] = round((current_assets - inventory) / current_liabilities, 2) if current_assets and current_liabilities and inventory else None
    results["Cash Ratio"] = round(cash / current_liabilities, 2) if cash and current_liabilities else None
 
    return results